# RQ2 RP-Geo — Stage-C finalize recovery (T4 x2)

Recovery-only notebook for a Save & Run whose RP-Geo training completed but finalization failed because one or more read-only variance diagnostics were absent. This notebook never calls a training function.

## Required inputs

1. The output of the failed Stage-C Save & Run, containing `e2e_pairwise_pilot_v2/resource_geo/checkpoints/epoch_100.pt`.
2. CIFAR-100 containing `cifar-100-python/{train,test,meta}`.
3. Gate-A output containing `gate_a_summary.json`.
4. Kaggle secret `github_token`. Enable T4 x2.

Do not attach an older Stage-C output at the same time.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib, shutil
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS = (0,1)
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Restore exactly the completed Stage-C tree

Selection is based on the frozen RP-Geo epoch-100 checkpoint, not merely on a similarly named folder.

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import scripts.run_e2e_pairwise_pilot as runner
pilot = importlib.reload(pilot); runner = importlib.reload(runner)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
candidates = []
for checkpoint in INPUT_ROOT.rglob('epoch_100.pt'):
    if checkpoint.parent.name != 'checkpoints' or checkpoint.parent.parent.name != 'resource_geo':
        continue
    root = checkpoint.parents[2]
    required = [root/'frozen_protocol.json', root/'resolved_config.yaml', root/'rpgeo_frozen_retention.json', root/'rpgeo_stage_c_protocol.json', root/'common_warmup/epoch_010.pt', root/'uniform/checkpoints/epoch_100.pt', root/'resource/checkpoints/epoch_100.pt', root/'pure_sw/checkpoints/epoch_100.pt', checkpoint]
    if all(path.is_file() for path in required): candidates.append(root)
candidates = sorted(set(candidates))
by_fingerprint = {}
for root in candidates:
    fingerprint = (pilot._sha256(root/'resource_geo/checkpoints/epoch_100.pt'), pilot._sha256(root/'rpgeo_frozen_retention.json'))
    by_fingerprint.setdefault(fingerprint, []).append(root)
assert len(by_fingerprint) == 1, f'Expected one content-unique completed Stage-C tree; found: {candidates}'
SOURCE_ROOT = sorted(next(iter(by_fingerprint.values())), key=lambda p:(len(str(p)),str(p)))[0]
ROOT = Path('/kaggle/working/e2e_pairwise_pilot_v2')
shutil.copytree(SOURCE_ROOT, ROOT, dirs_exist_ok=True)
freeze = pilot.load_frozen_rpgeo_retention(ROOT)
assert float(freeze['resource_retention']) == 0.995
provenance = json.loads((ROOT/'resource_geo/training_provenance.json').read_text())
assert float(provenance['resource_retention']) == 0.995
assert provenance['retention_frozen_before_e2e'] is True
print('Recovery source:', SOURCE_ROOT)
print('Working root:', ROOT)
print('CIFAR-100:', DATASET_ROOT)
print('Gate A:', GATE_A_SUMMARY)

## Complete only missing read-only diagnostics

No optimizer, scheduler, or model update occurs in this notebook.

In [ ]:
started = time.perf_counter()
missing_base = tuple(job for job in pilot.DIAGNOSTIC_STATES if job[0] != 'common_warmup' and not (ROOT/'diagnostics'/f'{job[0]}_E{job[1]}'/'variance.csv').is_file())
missing_frozen = tuple(job for job in (('common_warmup',10),('resource_geo',50),('resource_geo',100)) if not (ROOT/'diagnostics_rpgeo_frozen'/f'{job[0]}_E{job[1]}'/'variance.csv').is_file())
print('Missing base diagnostics:', missing_base)
print('Missing frozen diagnostics:', missing_frozen)
runtime_parts = []
if missing_base:
    runtime_parts.append(runner.run_diagnostics(ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS, jobs=missing_base))
if missing_frozen:
    runtime_parts.append(runner.run_diagnostics(ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS, jobs=missing_frozen, output_namespace='diagnostics_rpgeo_frozen'))
decision = pilot.finalize_rpgeo_extension(ROOT)
print(json.dumps(decision, indent=2))
for table in runtime_parts:
    if not table.empty: display(table)
display(__import__('pandas').read_csv(ROOT/'rpgeo_method_summary.csv'))
display(__import__('pandas').read_csv(ROOT/'rpgeo_frozen_policy_variance_t10_50_100.csv'))
print(f'Recovery completed in {(time.perf_counter()-started)/60:.1f} minutes')

## Export the complete resumable result

In [ ]:
required = ['rpgeo_frozen_retention.json','rpgeo_stage_c_protocol.json','resource_geo/checkpoints/epoch_100.pt','resource_geo/training_provenance.json','resource_geo/rpgeo_refresh_metrics.csv','rpgeo_summary.json','rpgeo_method_summary.csv','rpgeo_trajectory_variance_diagnostics.csv','rpgeo_frozen_policy_variance_t10_50_100.csv']
missing = [name for name in required if not (ROOT/name).is_file() or (ROOT/name).stat().st_size == 0]
assert not missing, f'Missing recovery artifacts: {missing}'
bundle = Path('/kaggle/working/rq2-rpgeo-frozen-e2e-v1-recovered.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print('Persist:',bundle,f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle